## 1. Housekeeping

In [1]:
import sqlite3
import csv
import urllib.request
import os

## 2. Download CSV Files

In [2]:
BOOK_PATH = "Book.csv"
MEMBER_PATH = "Member.csv"
LOAN_PATH = "Loan.csv"

DB_PATH = "library.db"

## 3. Create Database and Tables

In [3]:
print("Book exists:", os.path.exists(BOOK_PATH))
print("Member exists:", os.path.exists(MEMBER_PATH))
print("Loan exists:", os.path.exists(LOAN_PATH))

Book exists: True
Member exists: True
Loan exists: True


In [4]:
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

In [5]:
conn.execute("""
CREATE TABLE IF NOT EXISTS Book (
    callNo  TEXT    NOT NULL,
    title   TEXT    NOT NULL,
    author  TEXT    NOT NULL,
    PRIMARY KEY (callNo)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Member (
    id          INTEGER NOT NULL,
    firstname   TEXT    NOT NULL,
    lastName    TEXT    NOT NULL,
    PRIMARY KEY (id)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Loan (
    callNo        TEXT    NOT NULL,
    id            INTEGER NOT NULL,
    dateBorrowed  TEXT    NOT NULL,
    dateReturned  TEXT,
    dateDue       TEXT    NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id) REFERENCES Member(id)
);
""")

conn.commit()
print("Tables created.")

Tables created.


In [6]:
print(conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
).fetchall())

[('Book',), ('Loan',), ('Member',)]


## 4. Load Book Data

In [7]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);",
            (row["callNo"], row["title"], row["author"])
        )

conn.commit()

print("Book rows loaded:", conn.execute("SELECT COUNT(*) FROM Book;").fetchone()[0])

Book rows loaded: 11


## 5. Load Member Data

In [8]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);",
            (int(row["id"]), row["firstname"], row["lastName"])
        )

conn.commit()

print("Member rows loaded:", conn.execute("SELECT COUNT(*) FROM Member;").fetchone()[0])

Member rows loaded: 4


## 6. Load Loan Data

In [9]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        date_returned = row["dateReturned"] if row["dateReturned"].strip() else None

        conn.execute(
            """
            INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
            VALUES (?, ?, ?, ?, ?);
            """,
            (
                row["callNo"],
                int(row["id"]),
                row["dateBorrowed"],
                date_returned,
                row["dateDue"]
            )
        )

conn.commit()

print("Loan rows loaded:", conn.execute("SELECT COUNT(*) FROM Loan;").fetchone()[0])

Loan rows loaded: 4


In [10]:
print("Book count:", conn.execute("SELECT COUNT(*) FROM Book;").fetchone()[0])
print("Member count:", conn.execute("SELECT COUNT(*) FROM Member;").fetchone()[0])
print("Loan count:", conn.execute("SELECT COUNT(*) FROM Loan;").fetchone()[0])

Book count: 11
Member count: 4
Loan count: 4


## 7. ## Query 1 – All Books
Retrieve all columns from the Book table, ordered alphabetically by author.

In [11]:
query1 = """
SELECT *
FROM Book
ORDER BY author;
"""

rows = conn.execute(query1).fetchall()

for row in rows:
    print(row)

('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


## Query 2 – Books Not Yet Returned
Retrieve the title of each book, and the first and last name of the member who borrowed it, for all loans where dateReturned is NULL.

In [12]:
query2 = """
SELECT Book.title, Member.firstname, Member.lastName
FROM Loan
JOIN Book ON Loan.callNo = Book.callNo
JOIN Member ON Loan.id = Member.id
WHERE Loan.dateReturned IS NULL;
"""

rows = conn.execute(query2).fetchall()

for row in rows:
    print(row)

("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


## Query 3 – Loan History for a Specific Book
Retrieve the full loan history for the book with callNo R 141 E45 2006.
Show the member's full name, dateBorrowed, dateDue, and dateReturned.
Order by dateBorrowed ascending.

In [13]:
query3 = """
SELECT Member.firstname, Member.lastName, Loan.dateBorrowed, Loan.dateDue, Loan.dateReturned
FROM Loan
JOIN Member ON Loan.id = Member.id
WHERE Loan.callNo = 'R 141 E45 2006'
ORDER BY Loan.dateBorrowed ASC;
"""

rows = conn.execute(query3).fetchall()

for row in rows:
    print(row)

('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


## Query 4 – Members Who Have Never Borrowed a Book
Retrieve the id, firstname, and lastName of every member who does not appear in the Loan table.

In [14]:
query4 = """
SELECT Member.id, Member.firstname, Member.lastName
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
WHERE Loan.id IS NULL;
"""

rows = conn.execute(query4).fetchall()

for row in rows:
    print(row)

(4, 'John', 'Martin')


## Query 5 – Count of Loans per Member
Retrieve each member's full name and the total number of loans they have made, including completed ones.
Include members with zero loans.
Order by number of loans descending.

In [15]:
query5 = """
SELECT Member.firstname, Member.lastName, COUNT(Loan.callNo) AS loan_count
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
GROUP BY Member.id, Member.firstname, Member.lastName
ORDER BY loan_count DESC;
"""

rows = conn.execute(query5).fetchall()

for row in rows:
    print(row)

('David', 'Martin', 2)
('John', 'Smith', 1)
('Betty', 'Freeman', 1)
('John', 'Martin', 0)


## Query 6 – Custom Analytical Query
Business question: Which authors have been borrowed the most?
This is useful because it helps the library identify which authors are most popular with members.

In [16]:
query6 = """
SELECT Book.author, COUNT(*) AS borrow_count
FROM Loan
JOIN Book ON Loan.callNo = Book.callNo
GROUP BY Book.author
ORDER BY borrow_count DESC;
"""

rows = conn.execute(query6).fetchall()

for row in rows:
    print(row)

('Lynne Elliott', 2)
('Joe Celko', 1)
('Eileen Power', 1)


## Summary

One data quality issue in this dataset is that the dateReturned column contains empty values for books that have not yet been returned. These empty strings had to be converted to null before inserting the rows into SQLite. Another observation is that dates are stored as text rather than a standardized date type which limits accurate date comparisons. One limitation of this dataset is that it is very small and does not represent a real library system fully. A real library database would likely include multiple copies of books, fines, reservations, genres and more member details.

In [17]:
conn.close()
print("Database connection closed.")

Database connection closed.
